# Instance Segmentation: CNN vs Transformer × Training-Set Size

**Dataset:** `panoptic_indoor_dataset_segmentation` (922 images, 74 classes, YOLO polygon labels)
**Hardware:** Kaggle **GPU T4 ×2** (2 × 16 GB). Two experiments run in parallel, one per GPU.

| Role | What |
|---|---|
| **Independent variable** | training fraction of the train pool, `TRAIN_FRACTIONS` (placeholder, default `[0.25, 0.5, 1.0]`) |
| **Constants (all 4 models)** | image size, optimizer-step budget, effective batch, augmentation (h-flip + scale jitter), COCO-pretrained init, fp16 AMP, seed, warmup, eval protocol |
| **Model-specific constants** | optimizer / LR / schedule from each model's paper or framework recipe, frozen across all splits |
| **Dependent variables** | COCO mask metrics on the **fixed test set**: AP, AP50, AP75, APs/m/l, AR, AP(thing), AP(stuff), per-class AP, FPS |

| Model | Family | Paradigm |
|---|---|---|
| `yolo26l_seg` | CNN | one-stage, NMS-free (Ultralytics) |
| `maskrcnn_r50_fpn` | CNN | two-stage (torchvision) |
| `mask2former_swin_t` | Transformer | query-based, Swin-T backbone (HF transformers) |
| `rfdetr_seg_l` | Transformer | real-time DETR, DINOv2 backbone (rfdetr) |

**Splits.** Fixed once with `SEED`: val = 10 % (best-checkpoint selection), test = 10 % (final metrics), pool = remaining 80 %.
`train(f)` = first `f` of the shuffled pool, so the subsets are nested (25 % ⊂ 50 % ⊂ 100 %). Same val/test for every run.

**Order.** Model-major: model 1 × all splits, then model 2, …, two jobs at a time (GPU0, GPU1).

### How to run / resume (Save Version mode)
1. Settings: **Accelerator = GPU T4 ×2**, **Internet = On**. Add the dataset as input.
2. W&B: *Add-ons → Secrets* → label `WANDB_API_KEY` (recommended), or paste the key in the W&B cell.
3. **Save Version → Save & Run All (Commit)**, and keep *Always save output* on.
4. The run pauses itself before `SESSION_BUDGET_HOURS` (every worker stops at an epoch boundary, so a checkpoint is on disk), and the version's output is saved.
5. **To continue**, add the previous version's output as input (*Add Input → Your Work → this notebook*, or turn the output into a dataset), make sure it's the **latest** version, then Save Version again.
   Finished experiments are skipped. Interrupted ones resume from their last epoch. Remaining ones start.

### Outputs (`/kaggle/working/outputs`)
```
split_{pct}/{model}/best.*  last.*  metrics_test.json  per_class_ap_test.csv  status.json
results.csv        # dependent variables, one row per (model, split)
wandb_runs.csv     # one W&B run per experiment: id + URL + status
splits.json        # the fixed val / test / pool file lists
logs/{model}__split_{pct}.log
```
`/kaggle/working/runs/` holds in-progress training state, it is deleted per experiment when that experiment finishes.


## 1. Configuration (placeholders + constants)

In [ ]:
import time
SESSION_START_TS = time.time()

# ===================== INDEPENDENT VARIABLE (placeholder, edit me) =====================
TRAIN_FRACTIONS = [0.25, 0.5, 1.0]          # fractions of the train pool (80 % of the dataset)

# ===================== Session budget =====================
SESSION_BUDGET_HOURS = 11.0  # Kaggle hard limit is 12 h, set lower if your remaining weekly GPU quota is smaller
PAUSE_MARGIN_MIN = 40        # workers are told to pause this many minutes before the budget ends

# ===================== Paths =====================
DATASET_DIR = "/kaggle/input/datasets/jeremymattathiasmboe/panoptic-indoor-dataset-instance-segmentation"
WORKING = "/kaggle/working"

# ===================== Constants shared by ALL models (do not change between sessions) =====================
SHARED = dict(
    SEED=42,
    VAL_FRAC=0.10,                          # fixed val set (best-checkpoint selection)
    TEST_FRAC=0.10,                         # fixed test set (final metrics)
    IMG_SIZE=672,                           # long side, divisible by 32 (YOLO stride) and 24 (RF-DETR patch*windows)
    TRAIN_SCALES=list(range(552, 793, 24)), # scale jitter: long side in [552, 792] = IMG_SIZE x [0.82, 1.18]
    SCALE_JITTER=0.18,                      # same range expressed as Ultralytics `scale`
    HFLIP_P=0.5,                            # the only other augmentation
    EFFECTIVE_BATCH=8,                      # images per optimizer step (micro batch x grad accumulation)
    TOTAL_STEPS=4000,                       # optimizer-step budget per run -> epochs = ceil(TOTAL_STEPS / steps_per_epoch)
    NUM_EVALS=20,                           # ~validations per run (val every round(epochs/NUM_EVALS) epochs)
    WARMUP_STEPS=200,                       # linear LR warmup, in optimizer steps
    AMP=True,                               # fp16 mixed precision (T4 has no bf16)
    MAX_DET=100,                            # detections per image (COCO standard)
    SCORE_THRESH=0.001,                     # keep low-score detections for AP
)

# ===================== Model-specific recipes (constant across splits) =====================
# micro_batch only changes memory use (gradient accumulation keeps EFFECTIVE_BATCH), lower it on OOM.
MODELS = {
    "yolo26l_seg": dict(impl="yolo", family="CNN", weights="yolo26l-seg.pt", micro_batch=8,
                        # = Ultralytics optimizer='auto' choice for <10k iterations and nc=74, pinned explicitly
                        optimizer="AdamW", lr=0.000128, lrf=0.01, momentum=0.9, weight_decay=0.0005),
    "maskrcnn_r50_fpn": dict(impl="maskrcnn", family="CNN", micro_batch=8,
                             # Detectron/torchvision recipe: SGD 0.02 @ batch 16 -> linear scaling to batch 8
                             optimizer="SGD", lr=0.01, momentum=0.9, weight_decay=0.0001,
                             lr_milestones=[0.667, 0.889], lr_gamma=0.1),
    "mask2former_swin_t": dict(impl="mask2former", family="Transformer",
                               checkpoint="facebook/mask2former-swin-tiny-coco-instance", micro_batch=4,
                               # paper recipe: AdamW 1e-4, wd 0.05, backbone x0.1, poly 0.9, clip 0.01
                               optimizer="AdamW", lr=0.0001, weight_decay=0.05, backbone_lr_mult=0.1,
                               poly_power=0.9, clip_norm=0.01),
    "rfdetr_seg_l": {"impl": "rfdetr", "family": "Transformer", "class": "RFDETRSegLarge", "micro_batch": 4,
                     # rfdetr defaults: AdamW 1e-4 (encoder 1.5e-4), wd 1e-4, EMA, constant LR
                     "lr": 0.0001, "lr_encoder": 0.00015, "weight_decay": 0.0001},
}
MODEL_ORDER = ["yolo26l_seg", "maskrcnn_r50_fpn", "mask2former_swin_t", "rfdetr_seg_l"]
NUM_WORKERS = 2            # dataloader workers per job (Kaggle has few CPU cores, 2 jobs run at once)
MAX_ATTEMPTS = 3           # a crashing experiment is retried in later sessions up to this many times


## 2. Install dependencies

In [ ]:
import torch, torchvision
# keep Kaggle's CUDA build of torch, versions below are the ones this pipeline was tested with
open("/tmp/constraints.txt", "w").write(f"torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n")
!pip install -q -c /tmp/constraints.txt "ultralytics==8.4.31" "rfdetr[train]==1.12.1" "transformers==5.23.0" \
    "pytorch-lightning==2.6.6" "supervision==0.31.7" pycocotools wandb
!python -c "import ultralytics, rfdetr, transformers, pycocotools, torch"
!python -c "print('ultralytics', __import__('ultralytics').__version__, '| transformers', __import__('transformers').__version__, '| torch', __import__('torch').__version__, '| GPUs', __import__('torch').cuda.device_count())"


## 3. Weights & Biases API key

In [ ]:
# ===================== Weights & Biases =====================
# Recommended: Kaggle "Add-ons -> Secrets", label WANDB_API_KEY (the key is not stored in the notebook).
# Fallback: paste the key here. NOTE: a pasted key is saved inside this notebook version.
WANDB_API_KEY = ""                                 # <- optional manual paste
WANDB_PROJECT = "panoptic-indoor-instance-seg"
WANDB_ENTITY = ""                                  # team or username, empty = your default entity

import os
if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception as exc:
        print("Kaggle secret WANDB_API_KEY not available:", exc)
WANDB_ENABLED = bool(WANDB_API_KEY)
if WANDB_ENABLED:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("W&B enabled -> one run per experiment, links collected in outputs/wandb_runs.csv")
else:
    print("W&B DISABLED (no key). Training still runs, metrics are saved to outputs/.")


## 4. Pipeline source files
Each experiment runs in its own process (`worker.py`) pinned to one GPU.

In [ ]:
os.makedirs('/kaggle/working/src', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/src/common.py
"""Shared utilities: config, fixed splits, COCO conversion, unified COCO evaluation, run state, W&B."""
import contextlib
import csv
import io
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import yaml
from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXIT_PAUSED = 75  # worker stopped at an epoch boundary because the session deadline is near

# COCO-panoptic "stuff" categories present in this dataset (+ the stray "wall" class). Everything else is "thing".
STUFF_CLASSES = {
    "blanket", "cabinet-merged", "cardboard", "ceiling-merged", "counter", "curtain", "door-stuff",
    "floor-other-merged", "floor-wood", "flower", "food-other-merged", "light", "mirror-stuff", "paper-merged",
    "pillow", "rug-merged", "shelf", "stairs", "table-merged", "towel", "tree-merged", "wall", "wall-brick",
    "wall-other-merged", "wall-stone", "wall-tile", "wall-wood", "window-blind", "window-other",
}
METRIC_NAMES = ["segm_AP", "segm_AP50", "segm_AP75", "segm_APs", "segm_APm", "segm_APl",
                "segm_AR1", "segm_AR10", "segm_AR100", "segm_ARs", "segm_ARm", "segm_ARl"]


# ----------------------------------------------------------------------------- config / io
def load_config(path=None):
    return json.loads(Path(path or os.environ["EXP_CONFIG"]).read_text())


def atomic_write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str))
    os.replace(tmp, path)


def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default


def pct(frac):
    return int(round(float(frac) * 100))


def exp_name(model_key, frac):
    return f"{model_key}__split_{pct(frac)}"


def out_dir(cfg, model_key, frac):
    return Path(cfg["OUTPUT_DIR"]) / f"split_{pct(frac)}" / model_key


def run_dir(cfg, model_key, frac):
    return Path(cfg["RUNS_DIR"]) / model_key / f"split_{pct(frac)}"


def status_path(cfg, model_key, frac):
    return out_dir(cfg, model_key, frac) / "status.json"


def should_stop(cfg):
    return Path(cfg["STOP_FILE"]).exists()


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ----------------------------------------------------------------------------- schedule shared by all models
def schedule(cfg, n_train, micro_batch):
    """Fixed optimizer-step budget -> epochs for this split. Same rule for every model.

    steps_per_epoch counts optimizer steps at EFFECTIVE_BATCH (micro_batch * accumulation).
    Validation runs every `val_every_epochs` so every run gets ~NUM_EVALS validations regardless of split size.
    """
    eff = cfg["EFFECTIVE_BATCH"]
    assert eff % micro_batch == 0, f"EFFECTIVE_BATCH {eff} must be a multiple of micro batch {micro_batch}"
    steps_per_epoch = math.ceil(n_train / eff)
    epochs = math.ceil(cfg["TOTAL_STEPS"] / steps_per_epoch)
    return {
        "n_train": n_train,
        "steps_per_epoch": steps_per_epoch,
        "epochs": epochs,
        "total_steps": epochs * steps_per_epoch,
        "val_every_epochs": max(1, round(epochs / cfg["NUM_EVALS"])),
        "micro_batch": micro_batch,
        "accumulate": eff // micro_batch,
        "warmup_epochs": cfg["WARMUP_STEPS"] / steps_per_epoch,
    }


# ----------------------------------------------------------------------------- data
def class_names(cfg):
    names = yaml.safe_load((Path(cfg["DATASET_DIR"]) / "data.yaml").read_text())["names"]
    return list(names.values()) if isinstance(names, dict) else list(names)


def make_splits(cfg):
    """val/test fixed once (seeded) and persisted, train(frac) = first frac of the shuffled pool (nested)."""
    path = Path(cfg["OUTPUT_DIR"]) / "splits.json"
    if path.exists():
        return read_json(path)
    files = sorted(p.name for p in (Path(cfg["DATASET_DIR"]) / "images").iterdir() if p.suffix.lower() in IMG_EXTS)
    rng = random.Random(cfg["SEED"])
    rng.shuffle(files)
    n_val = round(len(files) * cfg["VAL_FRAC"])
    n_test = round(len(files) * cfg["TEST_FRAC"])
    splits = {
        "seed": cfg["SEED"],
        "val": sorted(files[:n_val]),
        "test": sorted(files[n_val:n_val + n_test]),
        "pool": files[n_val + n_test:],  # keep shuffled order -> nested train subsets
    }
    atomic_write_json(path, splits)
    return splits


def train_files(splits, frac):
    pool = splits["pool"]
    return sorted(pool[:max(1, round(len(pool) * float(frac)))])


def yolo_to_coco(cfg, files, names):
    img_dir = Path(cfg["DATASET_DIR"]) / "images"
    lbl_dir = Path(cfg["DATASET_DIR"]) / "labels"
    images, anns = [], []
    for img_id, fn in enumerate(files, 1):
        with Image.open(img_dir / fn) as im:
            w, h = im.size
        images.append({"id": img_id, "file_name": fn, "width": w, "height": h})
        lbl = lbl_dir / (Path(fn).stem + ".txt")
        for line in (lbl.read_text().splitlines() if lbl.exists() else []):
            vals = line.split()
            if len(vals) < 7 or (len(vals) - 1) % 2:
                continue  # fewer than 3 points or malformed
            xy = np.asarray(vals[1:], dtype=np.float64).reshape(-1, 2) * [w, h]
            xy[:, 0] = xy[:, 0].clip(0, w)
            xy[:, 1] = xy[:, 1].clip(0, h)
            poly = xy.flatten().tolist()
            area = float(mask_utils.area(mask_utils.merge(mask_utils.frPyObjects([poly], h, w))))
            if area < 1:
                continue
            x0, y0 = xy.min(0)
            x1, y1 = xy.max(0)
            anns.append({"id": len(anns) + 1, "image_id": img_id, "category_id": int(vals[0]) + 1,
                         "segmentation": [poly], "area": area,
                         "bbox": [float(x0), float(y0), float(x1 - x0), float(y1 - y0)], "iscrowd": 0})
    cats = [{"id": i + 1, "name": n, "supercategory": "stuff" if n in STUFF_CLASSES else "thing"}
            for i, n in enumerate(names)]
    return {"images": images, "annotations": anns, "categories": cats}


def _link(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        os.symlink(src, dst)


def prepare_data(cfg):
    """Build COCO json + YOLO dirs + RF-DETR dirs for every split. Idempotent, cheap to redo each session."""
    names = class_names(cfg)
    splits = make_splits(cfg)
    src_img = Path(cfg["DATASET_DIR"]) / "images"
    src_lbl = Path(cfg["DATASET_DIR"]) / "labels"
    data = Path(cfg["DATA_DIR"])
    sets = {"val": splits["val"], "test": splits["test"]}
    sets.update({f"train_{pct(f)}": train_files(splits, f) for f in cfg["TRAIN_FRACTIONS"]})

    for name, files in sets.items():
        cj = data / "coco" / f"{name}.json"
        if not cj.exists():
            atomic_write_json(cj, yolo_to_coco(cfg, files, names))
        ydir = data / "yolo" / name
        for fn in files:  # ultralytics maps .../images/x.jpg -> .../labels/x.txt by string replacement
            _link(src_img / fn, ydir / "images" / fn)
            lbl, dst = src_lbl / (Path(fn).stem + ".txt"), ydir / "labels" / (Path(fn).stem + ".txt")
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                if lbl.exists():
                    shutil.copy(lbl, dst)
                else:
                    dst.touch()

    for frac in cfg["TRAIN_FRACTIONS"]:
        p = pct(frac)
        (data / "yolo" / f"split_{p}.yaml").write_text(yaml.safe_dump({
            "path": str(data / "yolo"), "train": f"train_{p}/images", "val": "val/images",
            "names": dict(enumerate(names))}, sort_keys=False))
        for sub, name in (("train", f"train_{p}"), ("valid", "val"), ("test", "test")):
            rdir = data / "rfdetr" / f"split_{p}" / sub
            for fn in sets[name]:
                _link(src_img / fn, rdir / fn)
            shutil.copy(data / "coco" / f"{name}.json", rdir / "_annotations.coco.json")

    sizes = {k: len(v) for k, v in sets.items()}
    log(f"data ready: {sizes}")
    return {"names": names, "splits": splits, "sizes": sizes}


def coco_json(cfg, name):
    return Path(cfg["DATA_DIR"]) / "coco" / f"{name}.json"


def image_dir(cfg):
    return Path(cfg["DATASET_DIR"]) / "images"


def n_images(cfg, name):
    return len(read_json(coco_json(cfg, name))["images"])


# ----------------------------------------------------------------------------- unified evaluation
def coco_evaluate(gt_json, predict_fn, img_dir, max_det=100, fps_warmup=5, tag="eval"):
    """predict_fn(path) -> list of {"category_id", "score", "mask": bool HxW at original resolution}.

    Same pycocotools segm protocol for every model -> comparable dependent variables.
    """
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json))
    results, times = [], []
    img_ids = sorted(gt.getImgIds())
    for k, img_id in enumerate(img_ids):
        info = gt.imgs[img_id]
        t0 = time.perf_counter()
        preds = predict_fn(Path(img_dir) / info["file_name"])
        if k >= fps_warmup:
            times.append(time.perf_counter() - t0)
        for p in sorted(preds, key=lambda d: -d["score"])[:max_det]:
            m = np.asarray(p["mask"], dtype=np.uint8)
            assert m.shape == (info["height"], info["width"]), f"mask {m.shape} vs image {(info['height'], info['width'])}"
            if m.sum() == 0:
                continue
            rle = mask_utils.encode(np.asfortranarray(m))
            rle["counts"] = rle["counts"].decode("ascii")
            results.append({"image_id": img_id, "category_id": int(p["category_id"]),
                            "score": float(p["score"]), "segmentation": rle})
    fps = (len(times) / sum(times)) if times else float("nan")
    if not results:
        return {**{n: 0.0 for n in METRIC_NAMES}, "fps": fps, "num_images": len(img_ids)}, []
    with contextlib.redirect_stdout(io.StringIO()):
        ev = COCOeval(gt, gt.loadRes(results), "segm")
        ev.params.maxDets = [1, 10, max_det]
        ev.evaluate()
        ev.accumulate()
        ev.summarize()
    metrics = {n: float(v) for n, v in zip(METRIC_NAMES, ev.stats)}

    per_class = []
    prec = ev.eval["precision"]  # [T, R, K, A, M], A=0 -> all areas, M=-1 -> max_det
    for k, cid in enumerate(ev.params.catIds):
        p_all, p50 = prec[:, :, k, 0, -1], prec[0, :, k, 0, -1]
        cat = gt.cats[cid]
        per_class.append({"category_id": cid, "name": cat["name"], "type": cat["supercategory"],
                          "num_gt": len(gt.getAnnIds(catIds=[cid])),
                          "AP": float(p_all[p_all > -1].mean()) if (p_all > -1).any() else float("nan"),
                          "AP50": float(p50[p50 > -1].mean()) if (p50 > -1).any() else float("nan")})
    for kind in ("thing", "stuff"):
        aps = [c["AP"] for c in per_class if c["type"] == kind and c["num_gt"] > 0 and not math.isnan(c["AP"])]
        metrics[f"segm_AP_{kind}"] = float(np.mean(aps)) if aps else float("nan")
    metrics["classes_in_eval"] = sum(c["num_gt"] > 0 for c in per_class)
    metrics["fps"] = fps
    metrics["num_images"] = len(img_ids)
    log(f"{tag}: segm_AP={metrics['segm_AP']:.4f} AP50={metrics['segm_AP50']:.4f} fps={fps:.1f}")
    return metrics, per_class


# ----------------------------------------------------------------------------- W&B
def wandb_init(cfg, model_key, frac, status, sched):
    """One W&B run per experiment, stable id -> a resumed session appends to the same run."""
    if not cfg.get("WANDB_ENABLED"):
        return None
    import wandb
    run = wandb.init(
        project=cfg["WANDB_PROJECT"], entity=cfg.get("WANDB_ENTITY") or None,
        id=status["wandb_id"], resume="allow", name=exp_name(model_key, frac),
        group=f"split_{pct(frac)}", job_type=model_key,  # files go to $WANDB_DIR (not saved as output)
        config={"model": model_key, "train_fraction": frac, "schedule": sched,
                "shared": {k: cfg[k] for k in cfg["SHARED_KEYS"]}, "model_hp": cfg["MODELS"][model_key]},
    )
    status["wandb_url"] = run.url
    atomic_write_json(status_path(cfg, model_key, frac), status)
    return run


def wandb_log(data, step=None):
    try:
        import wandb
    except ImportError:
        return
    if wandb.run is not None:
        wandb.log(data, step=step)


def wandb_ensure(cfg, model_key, frac, status):
    """Some trainers (PTL) finish the run on exit, re-attach before logging test metrics."""
    if not cfg.get("WANDB_ENABLED"):
        return
    import wandb
    if wandb.run is None:
        wandb.init(project=cfg["WANDB_PROJECT"], entity=cfg.get("WANDB_ENTITY") or None,
                   id=status["wandb_id"], resume="allow")


def pause_and_exit():
    """Called at an epoch boundary (last checkpoint complete on disk) when the session deadline is near."""
    log("STOP file found -> pausing at epoch boundary")
    try:
        import wandb
        if wandb.run is not None:
            wandb.finish(exit_code=EXIT_PAUSED)
    except Exception as exc:  # never block the pause on logging
        log(f"wandb finish failed: {exc}")
    sys.stdout.flush()
    os._exit(EXIT_PAUSED)


# ----------------------------------------------------------------------------- finalize
def finalize(cfg, model_key, frac, status, best_src, last_src, metrics, per_class, extra=None):
    od = out_dir(cfg, model_key, frac)
    od.mkdir(parents=True, exist_ok=True)
    for src, stem in ((best_src, "best"), (last_src, "last")):
        if src and Path(src).exists():
            shutil.copy(src, od / f"{stem}{Path(src).suffix}")
    atomic_write_json(od / "metrics_test.json", {**metrics, **(extra or {})})
    with open(od / "per_class_ap_test.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["category_id", "name", "type", "num_gt", "AP", "AP50"])
        w.writeheader()
        w.writerows(per_class)

    wandb_ensure(cfg, model_key, frac, status)
    if cfg.get("WANDB_ENABLED"):
        import wandb
        if wandb.run is not None:
            wandb.run.summary.update({f"test/{k}": v for k, v in metrics.items()})
            if per_class:
                wandb.log({"test/per_class_AP": wandb.Table(columns=list(per_class[0].keys()),
                                                             data=[list(r.values()) for r in per_class])})
            status["wandb_url"] = wandb.run.url
            wandb.finish()

    status.update(status="done", finished_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                  test_segm_AP=metrics["segm_AP"], test_segm_AP50=metrics["segm_AP50"])
    atomic_write_json(status_path(cfg, model_key, frac), status)
    if cfg.get("CLEAN_RUN_DIRS", True):
        shutil.rmtree(run_dir(cfg, model_key, frac), ignore_errors=True)
    log(f"DONE {exp_name(model_key, frac)} -> {od}")


In [ ]:
%%writefile /kaggle/working/src/common_torch.py
"""Dataset + generic train loop shared by the two hand-written trainers (Mask R-CNN, Mask2Former)."""
import json
import math
import os
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps
from pycocotools import mask as mask_utils

import common as C


class CocoSegDataset(torch.utils.data.Dataset):
    """Shared augmentation spec: resize long side to a random TRAIN_SCALES value + horizontal flip (train only)."""

    def __init__(self, cfg, split_name, train):
        d = json.loads(C.coco_json(cfg, split_name).read_text())
        self.images = d["images"]
        self.anns = defaultdict(list)
        for a in d["annotations"]:
            self.anns[a["image_id"]].append(a)
        self.img_dir = C.image_dir(cfg)
        self.train = train
        self.scales = cfg["TRAIN_SCALES"]
        self.base = cfg["IMG_SIZE"]
        self.hflip_p = cfg["HFLIP_P"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        info = self.images[i]
        img = Image.open(self.img_dir / info["file_name"]).convert("RGB")
        w, h = img.size
        size = random.choice(self.scales) if self.train else self.base
        r = size / max(w, h)
        nw, nh = max(1, round(w * r)), max(1, round(h * r))
        img = img.resize((nw, nh), Image.Resampling.BILINEAR)
        flip = self.train and random.random() < self.hflip_p
        if flip:
            img = ImageOps.mirror(img)

        masks, labels = [], []
        for a in self.anns[info["id"]]:
            polys = []
            for poly in a["segmentation"]:
                xy = np.asarray(poly, dtype=np.float64).reshape(-1, 2) * r
                if flip:
                    xy[:, 0] = nw - xy[:, 0]
                polys.append(xy.flatten().tolist())
            m = mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(polys, nh, nw)))
            if m.any():
                masks.append(m)
                labels.append(a["category_id"] - 1)  # 0-based class index
        masks = torch.from_numpy(np.stack(masks)) if masks else torch.zeros((0, nh, nw), dtype=torch.uint8)
        return TF.to_tensor(img), {"masks": masks, "labels": torch.tensor(labels, dtype=torch.int64)}


def list_collate(batch):
    return tuple(zip(*batch))


def save_atomic(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def warmup_factor(cfg, step):
    w = cfg["WARMUP_STEPS"]
    return 1.0 if step >= w else 0.001 + (1 - 0.001) * step / w


def poly_lambda(cfg, power):
    def f(step, total):
        return warmup_factor(cfg, step) * max(0.0, 1 - step / total) ** power
    return f


def multistep_lambda(cfg, milestones_frac, gamma):
    def f(step, total):
        return warmup_factor(cfg, step) * gamma ** sum(step >= math.floor(m * total) for m in milestones_frac)
    return f


def _seed_worker(base):
    def init(worker_id):
        random.seed(base + worker_id)
        np.random.seed((base + worker_id) % 2**32)
    return init


def train_loop(cfg, model_key, frac, status, adapter):
    """AMP train loop: grad accumulation, epoch-boundary pause, periodic val (unified COCO eval), atomic ckpts.

    adapter provides: micro_batch, clip_norm, build(), param_groups(model), lr_lambda(step, total),
    loss(model, images, targets, device) -> (loss, logs dict), predictor(model, device) -> predict_fn(path).
    """
    device = torch.device("cuda")
    hp = cfg["MODELS"][model_key]
    rd = C.run_dir(cfg, model_key, frac)
    rd.mkdir(parents=True, exist_ok=True)
    state_ckpt, best_w, last_w = rd / "last_state.pt", rd / "best.pt", rd / "last.pt"

    ds = CocoSegDataset(cfg, f"train_{C.pct(frac)}", train=True)
    sched = C.schedule(cfg, len(ds), adapter.micro_batch)
    C.wandb_init(cfg, model_key, frac, status, sched)
    C.log(f"{C.exp_name(model_key, frac)} schedule {sched}")

    model = adapter.build().to(device)
    groups = adapter.param_groups(model)
    if hp["optimizer"] == "SGD":
        opt = torch.optim.SGD(groups, lr=hp["lr"], momentum=hp["momentum"], weight_decay=hp["weight_decay"])
    else:
        opt = torch.optim.AdamW(groups, lr=hp["lr"], weight_decay=hp["weight_decay"])
    total = sched["total_steps"]
    lr_sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: adapter.lr_lambda(s, total))
    scaler = torch.amp.GradScaler("cuda", enabled=cfg["AMP"])
    start_epoch, step, best_ap = 0, 0, -1.0

    if state_ckpt.exists():
        st = torch.load(state_ckpt, map_location="cpu", weights_only=False)
        model.load_state_dict(st["model"])
        opt.load_state_dict(st["optimizer"])
        lr_sched.load_state_dict(st["lr_sched"])
        scaler.load_state_dict(st["scaler"])
        start_epoch, step, best_ap = st["epoch"], st["step"], st["best_ap"]
        C.log(f"resumed at epoch {start_epoch}, step {step}, best val AP {best_ap:.4f}")

    accum = sched["accumulate"]
    for epoch in range(start_epoch, sched["epochs"]):
        if C.should_stop(cfg):
            C.pause_and_exit()  # state_ckpt of the previous epoch is complete
        g = torch.Generator().manual_seed(cfg["SEED"] + epoch)  # same order after resume
        loader = torch.utils.data.DataLoader(
            ds, batch_size=adapter.micro_batch, sampler=torch.utils.data.RandomSampler(ds, generator=g),
            num_workers=cfg["NUM_WORKERS"], collate_fn=list_collate, pin_memory=True, drop_last=False,
            worker_init_fn=_seed_worker(cfg["SEED"] * 10_000 + epoch * 100), persistent_workers=False)
        model.train()
        n_micro = len(loader)
        for i, (images, targets) in enumerate(loader):
            with torch.autocast("cuda", dtype=torch.float16, enabled=cfg["AMP"]):
                loss, logs = adapter.loss(model, images, targets, device)
            scaler.scale(loss / accum).backward()
            if (i + 1) % accum and (i + 1) != n_micro:
                continue
            if adapter.clip_norm:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), adapter.clip_norm)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            lr_sched.step()
            step += 1
            if step % cfg["LOG_EVERY_STEPS"] == 0:
                C.wandb_log({"train/loss": float(loss), **{f"train/{k}": v for k, v in logs.items()},
                             "train/lr": opt.param_groups[0]["lr"], "train/epoch": epoch + 1}, step=step)

        if (epoch + 1) % sched["val_every_epochs"] == 0 or epoch + 1 == sched["epochs"]:
            model.eval()
            with torch.no_grad():
                m, _ = C.coco_evaluate(C.coco_json(cfg, "val"), adapter.predictor(model, device), C.image_dir(cfg),
                                       max_det=cfg["MAX_DET"], tag=f"val ep{epoch + 1}/{sched['epochs']}")
            C.wandb_log({**{f"val/{k}": v for k, v in m.items()}, "train/epoch": epoch + 1}, step=step)
            if m["segm_AP"] > best_ap:
                best_ap = m["segm_AP"]
                save_atomic({"model": model.state_dict(), "epoch": epoch + 1, "val_segm_AP": best_ap}, best_w)
        save_atomic({"model": model.state_dict(), "optimizer": opt.state_dict(), "lr_sched": lr_sched.state_dict(),
                     "scaler": scaler.state_dict(), "epoch": epoch + 1, "step": step, "best_ap": best_ap},
                    state_ckpt)

    if not last_w.exists():
        save_atomic({"model": model.state_dict(), "epoch": sched["epochs"]}, last_w)
    model.load_state_dict(torch.load(best_w, map_location="cpu", weights_only=False)["model"])
    model.eval()
    return model, best_w, last_w, sched, best_ap


def evaluate_test(cfg, model_key, frac, status, predict_fn, best_w, last_w, sched, best_val):
    with torch.no_grad():
        metrics, per_class = C.coco_evaluate(C.coco_json(cfg, "test"), predict_fn, C.image_dir(cfg),
                                             max_det=cfg["MAX_DET"], tag="TEST")
    C.finalize(cfg, model_key, frac, status, best_w, last_w, metrics, per_class,
               extra={"best_val_segm_AP": best_val, "schedule": sched})


In [ ]:
%%writefile /kaggle/working/src/train_yolo.py
"""YOLO26-L-seg (Ultralytics, COCO-pretrained) -- CNN, one-stage, NMS-free."""
from pathlib import Path

import torch

import common as C


def _training_finished(last_pt):
    """Ultralytics strips the optimizer and sets epoch=-1 in last.pt once training completes."""
    return last_pt.exists() and torch.load(last_pt, map_location="cpu", weights_only=False).get("epoch", 0) == -1


def _add_callbacks(model, cfg, sched):
    validated = {"epoch": False}

    def on_train_epoch_start(trainer):
        if C.should_stop(cfg):
            C.pause_and_exit()  # last.pt of the previous epoch is complete

    def on_train_epoch_end(trainer):
        e = trainer.epoch + 1
        do_val = e % sched["val_every_epochs"] == 0 or e >= trainer.epochs
        trainer.args.val = do_val
        validated["epoch"] = do_val
        if not do_val:
            trainer.fitness = None  # otherwise save_model() would overwrite best.pt with unvalidated weights

    def on_fit_epoch_end(trainer):
        e = trainer.epoch + 1
        logs = {f"train/{k.split('/')[-1]}": float(v)
                for k, v in trainer.label_loss_items(trainer.tloss, prefix="train").items()}
        logs.update({"train/lr": trainer.lr.get("lr/pg0", 0.0), "train/epoch": e})
        if validated["epoch"]:
            logs.update({f"val_ultralytics/{k.split('/', 1)[-1]}": float(v) for k, v in trainer.metrics.items()})
        C.wandb_log(logs, step=e * sched["steps_per_epoch"])

    model.add_callback("on_train_epoch_start", on_train_epoch_start)
    model.add_callback("on_train_epoch_end", on_train_epoch_end)
    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)


def run(cfg, model_key, frac, status):
    from ultralytics import YOLO, settings
    if settings.get("wandb"):  # normally already disabled once by prefetch.py (avoids 2 workers writing at once)
        settings.update({"wandb": False})  # we log ourselves -> same W&B layout as the other models

    hp = cfg["MODELS"][model_key]
    p = C.pct(frac)
    rd = C.run_dir(cfg, model_key, frac)
    last_pt, best_pt = rd / "weights" / "last.pt", rd / "weights" / "best.pt"
    sched = C.schedule(cfg, C.n_images(cfg, f"train_{p}"), hp["micro_batch"])
    assert sched["accumulate"] == 1, "YOLO micro batch must equal EFFECTIVE_BATCH (nbs handles the rest)"
    C.wandb_init(cfg, model_key, frac, status, sched)
    C.log(f"{C.exp_name(model_key, frac)} schedule {sched}")

    if not _training_finished(last_pt):
        resume = last_pt.exists()
        model = YOLO(str(last_pt) if resume else hp["weights"])
        _add_callbacks(model, cfg, sched)
        if resume:
            C.log(f"resuming from {last_pt}")
            model.train(resume=True)
        else:
            model.train(
                data=str(Path(cfg["DATA_DIR"]) / "yolo" / f"split_{p}.yaml"),
                project=str(rd.parent), name=rd.name, exist_ok=True, device=0,
                epochs=sched["epochs"], imgsz=cfg["IMG_SIZE"], batch=hp["micro_batch"], nbs=cfg["EFFECTIVE_BATCH"],
                optimizer=hp["optimizer"], lr0=hp["lr"], lrf=hp["lrf"], momentum=hp["momentum"],
                weight_decay=hp["weight_decay"], warmup_epochs=sched["warmup_epochs"], cos_lr=False,
                # shared augmentation spec: horizontal flip + scale jitter only
                fliplr=cfg["HFLIP_P"], scale=cfg["SCALE_JITTER"], mosaic=0.0, close_mosaic=0, mixup=0.0,
                cutmix=0.0, copy_paste=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0,
                shear=0.0, perspective=0.0, flipud=0.0, bgr=0.0, multi_scale=0.0,
                amp=cfg["AMP"], seed=cfg["SEED"], deterministic=False, workers=cfg["NUM_WORKERS"],
                patience=0, max_det=cfg["MAX_DET"], plots=False, val=True, save=True)

    best = YOLO(str(best_pt))
    best_val = torch.load(best_pt, map_location="cpu", weights_only=False).get("train_metrics", {})

    def predict(path):
        r = best.predict(str(path), imgsz=cfg["IMG_SIZE"], conf=cfg["SCORE_THRESH"], max_det=cfg["MAX_DET"],
                         retina_masks=True, half=cfg["AMP"], device=0, verbose=False)[0]
        if r.masks is None:
            return []
        masks = (r.masks.data > 0.5).cpu().numpy()
        return [{"category_id": int(c) + 1, "score": float(s), "mask": masks[i]}
                for i, (c, s) in enumerate(zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()))]

    with torch.no_grad():
        metrics, per_class = C.coco_evaluate(C.coco_json(cfg, "test"), predict, C.image_dir(cfg),
                                             max_det=cfg["MAX_DET"], tag="TEST")
    C.finalize(cfg, model_key, frac, status, best_pt, last_pt, metrics, per_class,
               extra={"best_val_ultralytics": {k: float(v) for k, v in best_val.items()}, "schedule": sched})


In [ ]:
%%writefile /kaggle/working/src/train_maskrcnn.py
"""Mask R-CNN R50-FPN (torchvision, COCO-pretrained) -- CNN, two-stage."""
import torch
import torchvision
import torchvision.transforms.functional as TF
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

import common as C
import common_torch as CT


class MaskRCNNAdapter:
    def __init__(self, cfg, hp):
        self.cfg, self.hp = cfg, hp
        self.num_classes = len(C.class_names(cfg))
        self.micro_batch = hp["micro_batch"]
        self.clip_norm = None
        self.lr_lambda = CT.multistep_lambda(cfg, hp["lr_milestones"], hp["lr_gamma"])

    def build(self):
        weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.COCO_V1
        m = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=weights)
        nc = self.num_classes + 1  # + background
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, nc)
        m.roi_heads.mask_predictor = MaskRCNNPredictor(256, 256, nc)
        # Train images arrive already resized by the shared dataset, eval resizes long side to IMG_SIZE.
        m.transform._skip_resize = True  # torchvision only consults this in train mode
        m.transform.min_size = (self.cfg["IMG_SIZE"],)
        m.transform.max_size = self.cfg["IMG_SIZE"]
        m.roi_heads.score_thresh = self.cfg["SCORE_THRESH"]
        m.roi_heads.detections_per_img = self.cfg["MAX_DET"]
        return m

    def param_groups(self, model):
        decay, no_decay = [], []
        for n, p in model.named_parameters():
            if p.requires_grad:
                (no_decay if n.endswith(".bias") else decay).append(p)
        return [{"params": decay}, {"params": no_decay, "weight_decay": 0.0}]

    def loss(self, model, images, targets, device):
        imgs, tgts = [], []
        for img, t in zip(images, targets):
            masks = t["masks"]
            keep, boxes = [], []
            for k in range(masks.shape[0]):
                ys, xs = torch.where(masks[k] > 0)
                keep.append(k)
                boxes.append(torch.stack([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]).float())
            imgs.append(img.to(device, non_blocking=True))
            tgts.append({
                "boxes": torch.stack(boxes).to(device) if boxes else torch.zeros((0, 4), device=device),
                "labels": (t["labels"][keep] + 1).to(device),  # 1..num_classes (0 = background)
                "masks": masks[keep].to(device),
            })
        losses = model(imgs, tgts)
        return sum(losses.values()), {k: float(v) for k, v in losses.items()}

    def predictor(self, model, device):
        def predict(path):
            img = Image.open(path).convert("RGB")
            with torch.autocast("cuda", dtype=torch.float16, enabled=self.cfg["AMP"]):
                out = model([TF.to_tensor(img).to(device)])[0]
            masks = (out["masks"][:, 0] > 0.5).cpu().numpy()
            # torchvision labels 1..N == our COCO category ids
            return [{"category_id": int(l), "score": float(s), "mask": masks[i]}
                    for i, (l, s) in enumerate(zip(out["labels"].tolist(), out["scores"].tolist()))]
        return predict


def run(cfg, model_key, frac, status):
    adapter = MaskRCNNAdapter(cfg, cfg["MODELS"][model_key])
    model, best_w, last_w, sched, best_val = CT.train_loop(cfg, model_key, frac, status, adapter)
    CT.evaluate_test(cfg, model_key, frac, status, adapter.predictor(model, torch.device("cuda")),
                     best_w, last_w, sched, best_val)


In [ ]:
%%writefile /kaggle/working/src/train_mask2former.py
"""Mask2Former Swin-T (HF transformers, COCO-instance pretrained) -- transformer, query-based."""
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image

import common as C
import common_torch as CT

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
PAD_DIV = 32
MASK_CHUNK = 16


def _pad_batch(images, device):
    """Normalize, pad to a common size divisible by 32, pixel_mask marks valid pixels."""
    hmax = -(-max(i.shape[1] for i in images) // PAD_DIV) * PAD_DIV
    wmax = -(-max(i.shape[2] for i in images) // PAD_DIV) * PAD_DIV
    pv = torch.zeros((len(images), 3, hmax, wmax), device=device)
    pm = torch.zeros((len(images), hmax, wmax), dtype=torch.long, device=device)
    mean, std = MEAN.to(device), STD.to(device)
    for k, img in enumerate(images):
        _, h, w = img.shape
        pv[k, :, :h, :w] = (img.to(device) - mean) / std
        pm[k, :h, :w] = 1
    return pv, pm


class Mask2FormerAdapter:
    def __init__(self, cfg, hp):
        self.cfg, self.hp = cfg, hp
        self.names = C.class_names(cfg)
        self.micro_batch = hp["micro_batch"]
        self.clip_norm = hp["clip_norm"]
        self.lr_lambda = CT.poly_lambda(cfg, hp["poly_power"])

    def build(self):
        from transformers import Mask2FormerForUniversalSegmentation
        return Mask2FormerForUniversalSegmentation.from_pretrained(
            self.hp["checkpoint"], num_labels=len(self.names),
            id2label=dict(enumerate(self.names)), label2id={n: i for i, n in enumerate(self.names)},
            ignore_mismatched_sizes=True)  # new class head, everything else COCO-pretrained

    def param_groups(self, model):
        """Paper recipe: backbone lr x0.1, no weight decay on norms / biases / embeddings."""
        groups = {}
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            backbone = ".pixel_level_module.encoder." in f".{n}"
            no_wd = p.ndim == 1 or "embed" in n or "queries" in n or "norm" in n
            g = groups.setdefault((backbone, no_wd), {
                "params": [],
                "lr": self.hp["lr"] * (self.hp["backbone_lr_mult"] if backbone else 1.0),
                "weight_decay": 0.0 if no_wd else self.hp["weight_decay"]})
            g["params"].append(p)
        return list(groups.values())

    def loss(self, model, images, targets, device):
        pv, pm = _pad_batch(images, device)
        hp_, wp_ = pv.shape[-2:]
        mask_labels, class_labels = [], []
        for t in targets:
            m = t["masks"].to(device).float()
            mask_labels.append(F.pad(m, (0, wp_ - m.shape[-1], 0, hp_ - m.shape[-2])))
            class_labels.append(t["labels"].to(device))
        out = model(pixel_values=pv, pixel_mask=pm, mask_labels=mask_labels, class_labels=class_labels)
        return out.loss, {}

    def predictor(self, model, device):
        """Standard Mask2Former instance inference: top-k over (query, class), score = cls prob x mask quality."""
        size, max_det = self.cfg["IMG_SIZE"], self.cfg["MAX_DET"]

        def predict(path):
            img = Image.open(path).convert("RGB")
            w0, h0 = img.size
            r = size / max(w0, h0)
            nw, nh = max(1, round(w0 * r)), max(1, round(h0 * r))
            pv, pm = _pad_batch([TF.to_tensor(img.resize((nw, nh), Image.Resampling.BILINEAR))], device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=self.cfg["AMP"]):
                out = model(pixel_values=pv, pixel_mask=pm)
            cls = out.class_queries_logits[0].float().softmax(-1)[:, :-1]  # (Q, C) without "no object"
            n_cls = cls.shape[1]
            scores, idx = cls.flatten().topk(min(max_det, cls.numel()))
            q, labels = idx // n_cls, idx % n_cls
            ml = out.masks_queries_logits[0][q].float()[None]  # (1, K, h/4, w/4)
            ml = F.interpolate(ml, size=pv.shape[-2:], mode="bilinear", align_corners=False)[0, :, :nh, :nw]
            binm = ml > 0  # mask quality scored at network resolution (IMG_SIZE)
            mask_score = (ml.sigmoid() * binm).flatten(1).sum(1) / binm.flatten(1).sum(1).clamp(min=1)
            final = (scores * mask_score).tolist()
            # upsample to the original image in chunks: 100 full-res fp32 masks would not fit in memory
            binm = torch.cat([(F.interpolate(c[None], size=(h0, w0), mode="bilinear", align_corners=False)[0] > 0).cpu()
                              for c in ml.split(MASK_CHUNK)]).numpy()
            return [{"category_id": int(l) + 1, "score": s, "mask": binm[k]}
                    for k, (l, s) in enumerate(zip(labels.tolist(), final))]
        return predict


def run(cfg, model_key, frac, status):
    adapter = Mask2FormerAdapter(cfg, cfg["MODELS"][model_key])
    model, best_w, last_w, sched, best_val = CT.train_loop(cfg, model_key, frac, status, adapter)
    CT.evaluate_test(cfg, model_key, frac, status, adapter.predictor(model, torch.device("cuda")),
                     best_w, last_w, sched, best_val)


In [ ]:
%%writefile /kaggle/working/src/train_rfdetr.py
"""RF-DETR-Seg (Roboflow rfdetr, COCO-pretrained, DINOv2 backbone) -- transformer, real-time."""
from pathlib import Path

import numpy as np
import torch

import common as C


def _install_trainer_hooks(cfg):
    """rfdetr builds its PTL Trainer internally, inject our epoch-boundary pause callback via extra_callbacks."""
    import rfdetr.training.trainer as T
    from pytorch_lightning import Callback

    class PauseAtEpochStart(Callback):
        def on_train_epoch_start(self, trainer, pl_module):
            if C.should_stop(cfg):
                C.pause_and_exit()  # last.ckpt of the previous epoch is complete

    original = T.build_trainer
    if getattr(original, "_patched", False):
        return

    def build_trainer(*args, extra_callbacks=(), **kwargs):
        return original(*args, extra_callbacks=(*extra_callbacks, PauseAtEpochStart()), **kwargs)

    build_trainer._patched = True
    T.build_trainer = build_trainer
    import rfdetr.detr as D
    import rfdetr.training as TR
    for mod in (D, TR):
        if hasattr(mod, "build_trainer"):
            mod.build_trainer = build_trainer


def run(cfg, model_key, frac, status):
    import rfdetr

    hp = cfg["MODELS"][model_key]
    model_cls = getattr(rfdetr, hp["class"])
    p = C.pct(frac)
    rd = C.run_dir(cfg, model_key, frac)
    rd.mkdir(parents=True, exist_ok=True)
    last_ckpt, done_flag = rd / "last.ckpt", rd / "train_complete.json"
    sched = C.schedule(cfg, C.n_images(cfg, f"train_{p}"), hp["micro_batch"])
    C.wandb_init(cfg, model_key, frac, status, sched)  # PTL WandbLogger re-uses this active run
    C.log(f"{C.exp_name(model_key, frac)} schedule {sched}")

    if not done_flag.exists():
        _install_trainer_hooks(cfg)
        resume = str(last_ckpt) if last_ckpt.exists() else None
        if resume:
            C.log(f"resuming from {last_ckpt}")
        model = model_cls()
        model.train(
            dataset_dir=str(Path(cfg["DATA_DIR"]) / "rfdetr" / f"split_{p}"), dataset_file="roboflow",
            output_dir=str(rd), resume=resume,
            epochs=sched["epochs"], batch_size=hp["micro_batch"], grad_accum_steps=sched["accumulate"],
            lr=hp["lr"], lr_encoder=hp["lr_encoder"], weight_decay=hp["weight_decay"],
            warmup_epochs=sched["warmup_epochs"], lr_scheduler="step",
            lr_scheduler_kwargs={"lr_drop": sched["epochs"]},  # recipe default = constant LR (+EMA), no drop
            resolution=cfg["IMG_SIZE"],
            # shared augmentation spec: hflip (rfdetr default) + per-batch multi-scale, no random crop
            multi_scale=True, expanded_scales=True, scale_jitter=False, aug_config=None,
            eval_interval=sched["val_every_epochs"], checkpoint_interval=10**6, eval_max_dets=cfg["MAX_DET"],
            seed=cfg["SEED"], num_workers=cfg["NUM_WORKERS"], amp_dtype="fp16" if cfg["AMP"] else None,
            wandb=bool(cfg.get("WANDB_ENABLED")), project=cfg["WANDB_PROJECT"], run=C.exp_name(model_key, frac),
            tensorboard=False, progress_bar=None, early_stopping=False, run_test=False)
        C.atomic_write_json(done_flag, {"epochs": sched["epochs"]})

    best_pth = rd / "checkpoint_best_total.pth"
    model = model_cls.from_checkpoint(str(best_pth))
    names = C.class_names(cfg)
    cls_names = model.class_names
    cls_names = [cls_names[k] for k in sorted(cls_names)] if isinstance(cls_names, dict) else list(cls_names)
    name_to_cat = {n: i + 1 for i, n in enumerate(names)}
    idx_to_cat = {i: name_to_cat[n] for i, n in enumerate(cls_names) if n in name_to_cat}
    assert len(idx_to_cat) == len(names), f"class mapping mismatch: {cls_names[:5]}..."

    def predict(path):
        from PIL import Image
        det = model.predict(Image.open(path).convert("RGB"), threshold=cfg["SCORE_THRESH"])
        if det.mask is None or len(det) == 0:
            return []
        return [{"category_id": idx_to_cat[int(c)], "score": float(s), "mask": np.asarray(m, dtype=bool)}
                for c, s, m in zip(det.class_id, det.confidence, det.mask) if int(c) in idx_to_cat]

    with torch.no_grad():
        metrics, per_class = C.coco_evaluate(C.coco_json(cfg, "test"), predict, C.image_dir(cfg),
                                             max_det=cfg["MAX_DET"], tag="TEST")
    last_w = rd / "last_weights.pth"
    if last_ckpt.exists():  # weights only, full PTL state (optimizer, loops) is not needed after training
        sd = torch.load(last_ckpt, map_location="cpu", weights_only=False).get("state_dict", {})
        torch.save({"state_dict": sd}, last_w)
    C.finalize(cfg, model_key, frac, status, best_pth, last_w, metrics, per_class, extra={"schedule": sched})


In [ ]:
%%writefile /kaggle/working/src/worker.py
"""Run ONE experiment (model x train fraction) on the single GPU exposed via CUDA_VISIBLE_DEVICES.

Exit codes: 0 = finished (test metrics written), 75 = paused at an epoch boundary, other = failure.
"""
import argparse
import importlib
import random

import numpy as np
import torch

import common as C

TRAINERS = {"yolo": "train_yolo", "maskrcnn": "train_maskrcnn", "mask2former": "train_mask2former",
            "rfdetr": "train_rfdetr"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--frac", type=float, required=True)
    args = ap.parse_args()

    cfg = C.load_config()
    random.seed(cfg["SEED"])
    np.random.seed(cfg["SEED"])
    torch.manual_seed(cfg["SEED"])
    status = C.read_json(C.status_path(cfg, args.model, args.frac))
    C.log(f"worker start {C.exp_name(args.model, args.frac)} on {torch.cuda.get_device_name(0)}")
    trainer = importlib.import_module(TRAINERS[cfg["MODELS"][args.model]["impl"]])
    trainer.run(cfg, args.model, args.frac, status)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/prefetch.py
"""Download every pretrained checkpoint once, sequentially, before parallel workers start (avoids download races)."""
import os

import common as C


def main():
    cfg = C.load_config()
    for key, hp in cfg["MODELS"].items():
        C.log(f"prefetch {key}")
        if hp["impl"] == "yolo":
            from ultralytics import YOLO, settings
            settings.update({"wandb": False})  # workers log to W&B themselves
            YOLO(hp["weights"])
        elif hp["impl"] == "maskrcnn":
            import torchvision
            torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.COCO_V1.get_state_dict(progress=False)
        elif hp["impl"] == "mask2former":
            from huggingface_hub import snapshot_download
            snapshot_download(hp["checkpoint"])
        elif hp["impl"] == "rfdetr":
            import rfdetr
            getattr(rfdetr, hp["class"])()
    C.log(f"prefetch done (cwd={os.getcwd()})")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/orchestrator.py
"""Experiment queue (model-major, then split) executed on N GPUs in parallel, resumable across Kaggle sessions."""
import csv
import os
import shutil
import signal
import subprocess
import sys
import time
import uuid
from pathlib import Path

import common as C

SRC = Path(__file__).resolve().parent
POLL_SEC = 20
REPORT_EVERY_SEC = 600
KILL_GRACE_SEC = 60


# ----------------------------------------------------------------------------- state
def init_status(cfg, model_key, frac):
    path = C.status_path(cfg, model_key, frac)
    status = C.read_json(path)
    if status is None:
        status = {"model": model_key, "train_fraction": frac, "family": cfg["MODELS"][model_key]["family"],
                  "status": "pending", "attempts": 0, "wandb_id": uuid.uuid4().hex[:12], "wandb_url": ""}
        C.atomic_write_json(path, status)
    return status


def set_status(cfg, model_key, frac, **kw):
    status = C.read_json(C.status_path(cfg, model_key, frac))
    status.update(kw)
    C.atomic_write_json(C.status_path(cfg, model_key, frac), status)
    return status


def pending_jobs(cfg):
    jobs = []
    for model_key in cfg["MODEL_ORDER"]:  # loop: model, then training split
        for frac in cfg["TRAIN_FRACTIONS"]:
            st = init_status(cfg, model_key, frac)
            if st["status"] == "done":
                continue
            if st["status"] == "failed" and st["attempts"] >= cfg["MAX_ATTEMPTS"]:
                C.log(f"skip {C.exp_name(model_key, frac)}: failed {st['attempts']}x (see its log)")
                continue
            jobs.append((model_key, frac))
    return jobs


# ----------------------------------------------------------------------------- reports
def write_reports(cfg):
    out = Path(cfg["OUTPUT_DIR"])
    rows, results = [], []
    for model_key in cfg["MODEL_ORDER"]:
        for frac in cfg["TRAIN_FRACTIONS"]:
            st = C.read_json(C.status_path(cfg, model_key, frac)) or {}
            rows.append({"model": model_key, "family": cfg["MODELS"][model_key]["family"], "train_fraction": frac,
                         "status": st.get("status", "pending"), "attempts": st.get("attempts", 0),
                         "wandb_run_id": st.get("wandb_id", ""), "wandb_url": st.get("wandb_url", ""),
                         "finished_at": st.get("finished_at", "")})
            m = C.read_json(C.out_dir(cfg, model_key, frac) / "metrics_test.json")
            if m:
                sched = m.get("schedule", {})
                results.append({"model": model_key, "family": cfg["MODELS"][model_key]["family"],
                                "train_fraction": frac, "n_train": sched.get("n_train"),
                                "epochs": sched.get("epochs"), "optimizer_steps": sched.get("total_steps"),
                                **{k: m.get(k) for k in C.METRIC_NAMES + ["segm_AP_thing", "segm_AP_stuff",
                                                                         "classes_in_eval", "fps"]}})
    for name, data in (("wandb_runs.csv", rows), ("results.csv", results)):
        if data:
            with open(out / name, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(data[0].keys()))
                w.writeheader()
                w.writerows(data)
    return rows, results


def _tail(path, n=15):
    try:
        return "".join(Path(path).read_text(errors="replace").splitlines(keepends=True)[-n:])
    except OSError:
        return ""


# ----------------------------------------------------------------------------- runner
def _launch(cfg, gpu, model_key, frac):
    log_path = Path(cfg["OUTPUT_DIR"]) / "logs" / f"{C.exp_name(model_key, frac)}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    attempts = C.read_json(C.status_path(cfg, model_key, frac))["attempts"] + 1
    set_status(cfg, model_key, frac, status="running", gpu=gpu, attempts=attempts,
               started_at=time.strftime("%Y-%m-%d %H:%M:%S"))
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "EXP_CONFIG": cfg["CONFIG_PATH"],
           "PYTHONPATH": f"{SRC}{os.pathsep}{os.environ.get('PYTHONPATH', '')}", "PYTHONUNBUFFERED": "1",
           "WANDB_DIR": cfg["WANDB_DIR"], "WANDB_SILENT": "true"}
    Path(cfg["WORK_CWD"]).mkdir(parents=True, exist_ok=True)
    logf = open(log_path, "a")
    logf.write(f"\n===== session {time.strftime('%Y-%m-%d %H:%M:%S')} gpu {gpu} attempt {attempts} =====\n")
    logf.flush()
    proc = subprocess.Popen([sys.executable, str(SRC / "worker.py"), "--model", model_key, "--frac", str(frac)],
                            stdout=logf, stderr=subprocess.STDOUT, env=env, cwd=cfg["WORK_CWD"],
                            start_new_session=True)
    C.log(f"GPU{gpu} <- {C.exp_name(model_key, frac)} (attempt {attempts}, log {log_path.name})")
    return {"proc": proc, "job": (model_key, frac), "log": log_path, "logf": logf}


def _finish(cfg, slot):
    model_key, frac = slot["job"]
    code = slot["proc"].returncode
    slot["logf"].close()
    st = C.read_json(C.status_path(cfg, model_key, frac))
    if code == 0 and st["status"] == "done":
        C.log(f"finished {C.exp_name(model_key, frac)}  test segm_AP={st.get('test_segm_AP', float('nan')):.4f}")
    elif code == C.EXIT_PAUSED:
        set_status(cfg, model_key, frac, status="paused")
        C.log(f"paused {C.exp_name(model_key, frac)} (resumes next session)")
    else:
        set_status(cfg, model_key, frac, status="failed", last_exit_code=code, error_tail=_tail(slot["log"], 30))
        C.log(f"FAILED {C.exp_name(model_key, frac)} exit={code}\n{_tail(slot['log'])}")


def _kill(slot):
    try:
        os.killpg(slot["proc"].pid, signal.SIGTERM)
        slot["proc"].wait(KILL_GRACE_SEC)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        try:
            os.killpg(slot["proc"].pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        slot["proc"].wait()


def run_all(cfg):
    stop_file = Path(cfg["STOP_FILE"])
    stop_file.unlink(missing_ok=True)
    budget_end = cfg["SESSION_START_TS"] + cfg["SESSION_BUDGET_HOURS"] * 3600
    pause_at, kill_at = budget_end - cfg["PAUSE_MARGIN_MIN"] * 60, budget_end
    queue = pending_jobs(cfg)
    C.log(f"{len(queue)} experiment(s) to run: {[C.exp_name(*j) for j in queue]}")
    C.log(f"pause signal at {time.strftime('%H:%M:%S', time.localtime(pause_at))}, "
          f"hard stop at {time.strftime('%H:%M:%S', time.localtime(kill_at))}")
    slots = {gpu: None for gpu in cfg["GPUS"]}
    last_report = time.time()

    while queue or any(slots.values()):
        now = time.time()
        if now >= pause_at and not stop_file.exists():
            stop_file.touch()
            C.log("session budget nearly used -> workers pause at their next epoch boundary")
        for gpu in slots:
            slot = slots[gpu]
            if slot and slot["proc"].poll() is not None:
                _finish(cfg, slot)
                slots[gpu] = slot = None
            if slot is None and queue and not stop_file.exists():
                slots[gpu] = _launch(cfg, gpu, *queue.pop(0))
        if stop_file.exists() and not any(slots.values()):
            break  # paused everything, remaining queue runs next session
        if now >= kill_at:
            for gpu, slot in slots.items():
                if slot:
                    C.log(f"hard stop: killing {C.exp_name(*slot['job'])} (resumes from last epoch checkpoint)")
                    _kill(slot)
                    slot["proc"].returncode = C.EXIT_PAUSED
                    _finish(cfg, slot)
                    slots[gpu] = None
            break
        if now - last_report >= REPORT_EVERY_SEC:
            last_report = now
            for gpu, slot in slots.items():
                if slot:
                    C.log(f"GPU{gpu} {C.exp_name(*slot['job'])} ...\n{_tail(slot['log'], 4)}")
            write_reports(cfg)
        time.sleep(POLL_SEC)

    stop_file.unlink(missing_ok=True)
    rows, _ = write_reports(cfg)
    left = [r for r in rows if r["status"] != "done"]
    C.log("ALL EXPERIMENTS DONE" if not left else
          f"{len(left)} experiment(s) not done -> Save Version, attach this version's output as input, run again")
    return rows


# ----------------------------------------------------------------------------- cross-session restore
MARKER = "PIPELINE_MARKER.json"


def restore_previous_outputs(cfg):
    """Copy outputs/ + runs/ of a previous notebook version (attached as input) into the working dir."""
    out = Path(cfg["OUTPUT_DIR"])
    if (out / MARKER).exists():
        C.log("working dir already has pipeline state -> nothing to restore")
        return False
    candidates = sorted(Path(cfg["INPUT_ROOT"]).glob(f"**/{out.name}/{MARKER}"))
    if not candidates:
        C.log("no previous outputs attached -> starting fresh")
        return False
    prev_out = max(candidates, key=lambda p: C.read_json(p).get("updated", "")).parent
    prev_runs = prev_out.parent / Path(cfg["RUNS_DIR"]).name
    C.log(f"restoring previous state from {prev_out.parent}")
    shutil.copytree(prev_out, out, dirs_exist_ok=True)
    if prev_runs.exists():
        shutil.copytree(prev_runs, cfg["RUNS_DIR"], dirs_exist_ok=True)
    for st_path in out.glob("split_*/*/status.json"):  # anything interrupted mid-run resumes
        st = C.read_json(st_path)
        if st["status"] == "running":
            st["status"] = "paused"
            C.atomic_write_json(st_path, st)
    return True


def write_marker(cfg):
    path = Path(cfg["OUTPUT_DIR"]) / MARKER
    prev = C.read_json(path, {})
    C.atomic_write_json(path, {"created": prev.get("created", time.strftime("%Y-%m-%d %H:%M:%S")),
                               "updated": time.strftime("%Y-%m-%d %H:%M:%S"),
                               "shared_config": {k: cfg[k] for k in cfg["SHARED_KEYS"]}})


## 5. Config file, restore previous session, build splits

In [ ]:
import json, glob, sys
from pathlib import Path
import torch

SRC_DIR = f"{WORKING}/src"
if not Path(DATASET_DIR, "data.yaml").exists():  # fall back to wherever Kaggle mounted the dataset
    hits = [Path(p).parent for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True)
            if Path(p).parent.joinpath("images").is_dir() and Path(p).parent.joinpath("labels").is_dir()]
    assert hits, f"dataset not found at {DATASET_DIR} or anywhere under /kaggle/input"
    DATASET_DIR = str(hits[0])
print("dataset:", DATASET_DIR)

n_gpu = torch.cuda.device_count()
assert n_gpu >= 1, "No GPU visible: set Accelerator = GPU T4 x2"
cfg = dict(
    **SHARED, TRAIN_FRACTIONS=TRAIN_FRACTIONS, MODELS=MODELS, MODEL_ORDER=MODEL_ORDER,
    SHARED_KEYS=list(SHARED) + ["TRAIN_FRACTIONS"],
    NUM_WORKERS=NUM_WORKERS, MAX_ATTEMPTS=MAX_ATTEMPTS, LOG_EVERY_STEPS=20, CLEAN_RUN_DIRS=True,
    DATASET_DIR=DATASET_DIR, OUTPUT_DIR=f"{WORKING}/outputs", RUNS_DIR=f"{WORKING}/runs",
    DATA_DIR="/tmp/exp_data", WORK_CWD="/tmp/exp_cwd", WANDB_DIR="/tmp/wandb", STOP_FILE="/tmp/exp_STOP",
    INPUT_ROOT="/kaggle/input", CONFIG_PATH=f"{WORKING}/exp_config.json",
    WANDB_ENABLED=WANDB_ENABLED, WANDB_PROJECT=WANDB_PROJECT, WANDB_ENTITY=WANDB_ENTITY,
    GPUS=list(range(min(n_gpu, 2))), SESSION_START_TS=SESSION_START_TS,
    SESSION_BUDGET_HOURS=SESSION_BUDGET_HOURS, PAUSE_MARGIN_MIN=PAUSE_MARGIN_MIN,
)
for d in ("DATA_DIR", "WORK_CWD", "WANDB_DIR"):
    Path(cfg[d]).mkdir(parents=True, exist_ok=True)
Path(cfg["CONFIG_PATH"]).write_text(json.dumps(cfg, indent=1))
os.environ["EXP_CONFIG"] = cfg["CONFIG_PATH"]
sys.path.insert(0, SRC_DIR)
print(f"GPUs used: {cfg['GPUS']} | runs: {len(MODEL_ORDER)} models x {len(TRAIN_FRACTIONS)} splits")


In [ ]:
import common as C, orchestrator as O

O.restore_previous_outputs(cfg)      # copies outputs/ + runs/ from an attached previous version (if any)
prev = C.read_json(Path(cfg["OUTPUT_DIR"]) / "PIPELINE_MARKER.json")
if prev:  # guard: constants must not change between sessions, otherwise runs are not comparable
    changed = {k: (prev["shared_config"].get(k), cfg[k]) for k in cfg["SHARED_KEYS"]
               if prev["shared_config"].get(k) != cfg[k]}
    assert not changed, f"Shared constants changed since the first session: {changed}"
info = C.prepare_data(cfg)
O.write_marker(cfg)
for f in TRAIN_FRACTIONS:
    sched = C.schedule(cfg, info["sizes"][f"train_{C.pct(f)}"], cfg["EFFECTIVE_BATCH"])
    print(f"split {C.pct(f):>3}% -> {sched['n_train']} train imgs, {sched['epochs']} epochs, "
          f"{sched['total_steps']} steps, val every {sched['val_every_epochs']} epochs")


## 6. Pre-download pretrained weights

In [ ]:
import subprocess
# download all pretrained weights once, before two workers start in parallel
subprocess.run([sys.executable, f"{SRC_DIR}/prefetch.py"], check=True, cwd=cfg["WORK_CWD"],
               env={**os.environ, "EXP_CONFIG": cfg["CONFIG_PATH"],
                    "PYTHONPATH": os.pathsep.join(filter(None, [SRC_DIR, os.environ.get("PYTHONPATH")]))})


## 7. Train + evaluate all experiments (2 GPUs in parallel)

In [ ]:
# Runs every unfinished (model, split) pair, 2 at a time. Progress is printed every 10 min.
# Full per-experiment logs: outputs/logs/. Safe to re-run: finished experiments are skipped.
rows = O.run_all(cfg)


## 8. Results (dependent variables) and W&B links

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

O.write_marker(cfg)
out = Path(cfg["OUTPUT_DIR"])
display(pd.read_csv(out / "wandb_runs.csv"))
if (out / "results.csv").exists():
    res = pd.read_csv(out / "results.csv")
    display(res)
    for metric in ["segm_AP", "segm_AP50", "segm_AP_thing", "segm_AP_stuff"]:
        display(res.pivot(index="model", columns="train_fraction", values=metric).round(4).rename_axis(metric))
    fig, ax = plt.subplots(figsize=(7, 4))
    for model, g in res.groupby("model"):
        g = g.sort_values("train_fraction")
        ax.plot(g["train_fraction"], g["segm_AP"], marker="o", label=f"{model} ({g['family'].iloc[0]})")
    ax.set(xlabel="training fraction", ylabel="test mask AP (COCO)", title="Mask AP vs training-set size")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(out / "ap_vs_train_fraction.png", dpi=150)
    plt.show()
